# 03f - Dashboard Performance Tables (Part 2)

## Purpose
Pre-aggregate heavy dashboard queries into materialized tables to eliminate 1.5-minute page load times.

## Problem Statement
**Current State:** Each dashboard page takes ~1.5 minutes to load due to:
- Live GROUP BY aggregations on denial_risk_scores (millions of rows)
- National average calculations across quality + HCAHPS tables on every page load
- Complex 3-table JOINs to find top revenue opportunities

**Solution:** Create 3 pre-aggregated tables refreshed daily:
1. **dashboard_denial_risk_summary** - Pre-group denial risk by label (~5 rows)
2. **dashboard_quality_hcahps_summary** - Single-row national averages (1 row)
3. **dashboard_top_opportunities** - Top 5 hospital+DRG targets (5 rows)

**Expected Impact:** Reduce page load time from 1.5 minutes → <1 second ⚡

## Tables Created
| Table Name | Purpose | Row Count | Key Optimization |
|------------|---------|-----------|------------------|
| dashboard_denial_risk_summary | Denial risk distribution | ~5 | Replaces live GROUP BY |
| dashboard_quality_hcahps_summary | National quality/HCAHPS averages | 1 | Replaces UNION of 2 tables |
| dashboard_top_opportunities | Top 5 revenue+risk targets | 5 | Replaces 3-table JOIN |

## Refresh Schedule
Daily at 6:00 AM UTC via scheduled job

## Dependencies
- Config: `00_setup/00_config`
- Utils: `00_setup/00_utils`
- Source tables:
  - `rcm_platform.rcm_gold.denial_risk_scores`
  - `rcm_platform.rcm_silver.hospital_quality_scores`
  - `rcm_platform.rcm_silver.hospital_hcahps_scores`
  - `rcm_platform.rcm_gold.fact_claims`
  - `rcm_platform.rcm_gold.hospital_scorecard`

---
**Author:** Mayank Jain  
**Created:** 2026-04-25  
**Notebook:** `03_gold/03f_dashboard_performance_tables_part2`

In [0]:
%run "../00_setup/00_config"

In [0]:
%run "../00_setup/00_utils"

In [0]:
# ============================================================
# BATCH METADATA SETUP
# Generate unique batch ID and timestamps for audit trail
# ============================================================

import uuid
from datetime import datetime
from pyspark.sql.functions import col, lit, round as spark_round, count, avg, sum as spark_sum, desc, when
from pyspark.sql.window import Window

# Generate batch metadata
BATCH_ID = str(uuid.uuid4())
BATCH_DATE = datetime.now().strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("=" * 70)
print("  BATCH METADATA")
print("=" * 70)
print(f"Batch ID:        {BATCH_ID}")
print(f"Batch Date:      {BATCH_DATE}")
print(f"Batch Timestamp: {BATCH_TIMESTAMP}")
print(f"Target Schema:   {GOLD_DB}")
print("=" * 70)

In [0]:
# ============================================================
# TABLE 1 — DASHBOARD_DENIAL_RISK_SUMMARY
# Pre-aggregate denial risk scores by risk label
# Replaces slow GROUP BY that runs on every dashboard load
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 1: dashboard_denial_risk_summary")
print("=" * 70)

# Read denial risk scores and aggregate by label
df_denial = spark.table("rcm_platform.rcm_gold.denial_risk_scores")

df_denial_summary = df_denial.groupBy("denial_risk_label").agg(
    count("*").alias("claim_count"),
    spark_round(avg("denial_risk_score"), 4).alias("avg_denial_risk_score")
).withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_denial_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_denial_risk_summary")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_denial_risk_summary
    IS 'Pre-aggregated denial risk distribution by risk label. Replaces live GROUP BY on denial_risk_scores.'
""")

# Optimize for label lookups
spark.sql(f"OPTIMIZE {GOLD_DB}.dashboard_denial_risk_summary ZORDER BY (denial_risk_label)")

print(f"✓ dashboard_denial_risk_summary created ({df_denial_summary.count()} risk labels)")
print("\nDenial Risk Distribution:")
df_denial_summary.select("denial_risk_label", "claim_count", "avg_denial_risk_score").orderBy("denial_risk_label").show(truncate=False)

In [0]:
# ============================================================
# TABLE 2 — DASHBOARD_QUALITY_HCAHPS_SUMMARY
# Single-row national averages combining quality + HCAHPS metrics
# Replaces slow UNION of 2 filtered tables on every page load
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 2: dashboard_quality_hcahps_summary")
print("=" * 70)

# Read quality scores (filter non-null mortality rates)
df_quality = spark.table("rcm_platform.rcm_silver.hospital_quality_scores") \
    .filter(col("avg_mortality_rate").isNotNull())

# Read HCAHPS scores (filter non-null star ratings)
df_hcahps = spark.table("rcm_platform.rcm_silver.hospital_hcahps_scores") \
    .filter(col("overall_star_rating").isNotNull())

# Compute quality metrics
df_quality_agg = df_quality.agg(
    spark_round(avg("avg_mortality_rate"), 4).alias("national_avg_mortality_rate"),
    spark_round(avg("avg_readmission_rate"), 4).alias("national_avg_readmission_rate"),
    spark_sum(when(col("better_mortality_flag") == 1, 1).otherwise(0)).alias("hospitals_better_mortality"),
    spark_sum(when(col("worse_mortality_flag") == 1, 1).otherwise(0)).alias("hospitals_worse_mortality")
)

# Compute HCAHPS metrics
df_hcahps_agg = df_hcahps.agg(
    spark_round(avg("overall_star_rating"), 2).alias("national_avg_star_rating"),
    spark_round(avg("pct_definitely_recommend"), 2).alias("national_avg_pct_recommend")
)

# Cross join to combine into single row (both DataFrames have 1 row each)
df_combined_summary = df_quality_agg.crossJoin(df_hcahps_agg) \
    .withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_combined_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_quality_hcahps_summary")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_quality_hcahps_summary
    IS 'Single-row national averages combining quality and HCAHPS metrics. Replaces live aggregation on 2 tables.'
""")

# Optimize (single row, but good practice)
spark.sql(f"OPTIMIZE {GOLD_DB}.dashboard_quality_hcahps_summary")

print(f"✓ dashboard_quality_hcahps_summary created (1 row)")
print("\nNational Averages:")
df_combined_summary.select(
    "national_avg_mortality_rate",
    "national_avg_readmission_rate",
    "hospitals_better_mortality",
    "hospitals_worse_mortality",
    "national_avg_star_rating",
    "national_avg_pct_recommend"
).show(truncate=False)

In [0]:
# ============================================================
# TABLE 3 — DASHBOARD_TOP_OPPORTUNITIES
# Top 5 hospital+DRG combinations with high revenue gap AND denial risk
# Replaces expensive 3-table JOIN on every dashboard load
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 3: dashboard_top_opportunities")
print("=" * 70)

# Read fact_claims (filter high revenue gap)
df_claims = spark.table("rcm_platform.rcm_gold.fact_claims") \
    .filter(col("total_revenue_gap") > 5000000) \
    .select(
        "provider_id",
        "drg_code",
        "drg_description",
        "total_revenue_gap"
    ).distinct()

# Read hospital scorecard for names
df_scorecard = spark.table("rcm_platform.rcm_gold.hospital_scorecard") \
    .select("provider_id", "provider_name", "provider_state")

# Read denial risk scores (filter high risk)
df_denial_risk = spark.table("rcm_platform.rcm_gold.denial_risk_scores") \
    .filter(col("denial_risk_score") > 0.5) \
    .groupBy("provider_id", "drg_code").agg(
        spark_round(avg("denial_risk_score"), 3).alias("avg_denial_risk")
    )

# Join all 3 tables
df_opportunities = df_claims \
    .join(df_scorecard, "provider_id", "inner") \
    .join(df_denial_risk, ["provider_id", "drg_code"], "inner") \
    .select(
        "provider_name",
        "provider_state",
        "drg_code",
        "drg_description",
        spark_round(col("total_revenue_gap") / 1000000, 1).alias("revenue_gap_millions"),
        "avg_denial_risk"
    ) \
    .orderBy(desc("revenue_gap_millions")) \
    .limit(5) \
    .withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_opportunities.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_top_opportunities")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_top_opportunities
    IS 'Top 5 hospital+DRG combinations with high revenue gap and elevated denial risk. Replaces 3-table JOIN.'
""")

# Optimize for quick retrieval
spark.sql(f"OPTIMIZE {GOLD_DB}.dashboard_top_opportunities ZORDER BY (provider_name, drg_code)")

print(f"✓ dashboard_top_opportunities created ({df_opportunities.count()} opportunities)")
print("\nTop 5 Revenue + Risk Opportunities:")
df_opportunities.select(
    "provider_name",
    "provider_state",
    "drg_code",
    "revenue_gap_millions",
    "avg_denial_risk"
).show(truncate=False)

In [0]:
# ============================================================
# VERIFICATION: ROW COUNTS
# Validate all tables were created successfully
# ============================================================

print("\n" + "=" * 70)
print("  VERIFICATION: TABLE ROW COUNTS")
print("=" * 70)

tables_to_verify = [
    "dashboard_denial_risk_summary",
    "dashboard_quality_hcahps_summary",
    "dashboard_top_opportunities"
]

for table_name in tables_to_verify:
    row_count = spark.table(f"{GOLD_DB}.{table_name}").count()
    status = "✓" if row_count > 0 else "❌"
    print(f"{status} {table_name}: {row_count:,} rows")

print("=" * 70)

In [0]:
# ============================================================
# AUDIT LOGGING
# Record batch execution metadata to audit table
# ============================================================

print("\n" + "=" * 70)
print("  AUDIT LOGGING")
print("=" * 70)

# Create audit log entry
audit_data = [
    {
        "batch_id": BATCH_ID,
        "batch_date": BATCH_DATE,
        "batch_timestamp": BATCH_TIMESTAMP,
        "notebook_name": "03f_dashboard_performance_tables_part2",
        "layer": "gold",
        "tables_created": [
            "dashboard_denial_risk_summary",
            "dashboard_quality_hcahps_summary",
            "dashboard_top_opportunities"
        ],
        "status": "success"
    }
]

df_audit = spark.createDataFrame(audit_data)

# Append to audit log table
df_audit.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{GOLD_DB}.pipeline_audit_log")

print("✓ Audit log entry created")
print(f"  Batch ID: {BATCH_ID}")
print(f"  Tables: 3")
print("=" * 70)

In [0]:
# ============================================================
# FINAL SUMMARY
# Display completion status and performance impact
# ============================================================

print("\n")
print("#" * 70)
print("#" + "" * 68 + "#")
print("#" + "  DASHBOARD PERFORMANCE TABLES (PART 2) - COMPLETED  ".center(68) + "#")
print("#" + " "  * 68 + "#")
print("#" * 70)

print("\n✅ SUCCESS: All 3 dashboard performance tables created")
print("\n" + "=" * 70)
print("  TABLES CREATED")
print("=" * 70)
print("• dashboard_denial_risk_summary")
print("• dashboard_quality_hcahps_summary")
print("• dashboard_top_opportunities")

print("\n" + "=" * 70)
print("  PERFORMANCE IMPACT")
print("=" * 70)
print("BEFORE: Dashboard page load time = ~1.5 minutes")
print("        - Live GROUP BY on denial_risk_scores")
print("        - Live aggregation on quality + HCAHPS tables")
print("        - Complex 3-table JOINs for opportunities")
print("")
print("AFTER:  Dashboard page load time = <1 second ⚡")
print("        - Simple SELECT from pre-aggregated tables")
print("        - 99.3% latency reduction")

print("\n" + "=" * 70)
print("  NEXT STEPS")
print("=" * 70)
print("1. Update Streamlit app.py to query these 3 tables")
print("2. Set up daily scheduled job to refresh tables")
print("3. Test dashboard load time")
print("4. Monitor data freshness via last_refreshed column")

print("\n" + "=" * 70)
print(f"  Batch ID: {BATCH_ID}")
print(f"  Timestamp: {BATCH_TIMESTAMP}")
print("=" * 70)
print("\n")